# Iris Flower Measurements: A Reproducible Data Workflow

**Author:** Ewald Geschwinde

**Dataset:** [Iris](https://archive.ics.uci.edu/dataset/53/iris) (Fisher, 1936), UCI Machine Learning Repository

This project walks a classic tabular dataset through a complete, reproducible
analysis workflow: ingestion from a local CSV, cleaning with documented and
reusable functions, exploratory analysis, and visualization. The dataset records
four morphological measurements for 150 iris flowers drawn evenly from three
species, and the guiding question is how well those measurements distinguish one
species from another. Every step runs top-to-bottom without manual intervention
and without network access, so a reader can reproduce each number and figure
below from the committed CSV alone.

---

## Contents

1. Environment and imports
2. Data ingestion
3. Data cleaning
4. Exploratory data analysis
5. Visualizations
6. Summary and interpretation

## 1. Environment and imports

Library versions are printed deliberately. A notebook that records the versions
it ran under can be re-created later; one that does not leaves a reader guessing
which API a result depended on. The pinned versions also appear in
`requirements.txt`.

In [12]:
import hashlib
import io
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", plt.matplotlib.__version__)
print("seaborn    ", sns.__version__)

numpy       2.5.4
pandas      3.0.6
matplotlib  3.11.2
seaborn     0.13.2


### Shared plotting style

The figures later in the notebook share one visual system, defined once here so
no single chart invents its own look. Three choices matter:

- **A fixed categorical palette.** The three species each get a permanent colour
  that never changes between figures, so a reader who learns "setosa is blue" in
  Figure 1 can rely on it in Figure 4. The three hues were checked for
  colour-vision-deficiency separation rather than chosen by eye.
- **Marker shapes as a second channel.** Species is encoded by shape as well as
  colour, so the scatter plots stay readable in greyscale, in print, or to a
  reader with colour-vision deficiency. Colour is never the only carrier of
  meaning.
- **Recessive chrome.** Hairline gridlines and muted axis labels keep the data
  marks, not the scaffolding, as the darkest thing on the page.

In [13]:
SPECIES_ORDER = ["setosa", "versicolor", "virginica"]

# Fixed per-species encoding, reused by every figure in this notebook.
SPECIES_COLORS = {
    "setosa":     "#2a78d6",  # blue
    "versicolor": "#eb6834",  # orange
    "virginica":  "#1baf7a",  # aqua
}
SPECIES_MARKERS = {"setosa": "o", "versicolor": "s", "virginica": "^"}

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "figure.facecolor":  SURFACE,
    "axes.facecolor":    SURFACE,
    "axes.edgecolor":    GRID,
    "axes.labelcolor":   INK,
    "axes.titlecolor":   INK,
    "axes.titleweight":  "bold",
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.6,
    "grid.linestyle":    "-",
    "xtick.color":       MUTED,
    "ytick.color":       MUTED,
    "savefig.facecolor": SURFACE,
    "figure.dpi":        110,
})

## 2. Data ingestion

`iris.csv` sits beside this notebook and was produced from the UCI distribution
file `iris/iris.data` by adding a header row; no values were altered. The
analysis reads this committed copy rather than downloading the data each time it
runs, which is a deliberate reproducibility choice: a file in the repository
cannot change underneath the analysis, whereas a remote one can be revised or
withdrawn without anyone noticing. Keeping the raw download alongside the CSV
means the provenance of every number stays inspectable, and the cleaning steps in
Section 3 remain visible in code rather than being silently baked into the input.

That choice has a cost, which is that the local copy could silently fall behind
the published dataset. Section 2.1 therefore provides a documented way to check
the local file against the source and refresh it, so the convenience of a
committed file does not become a reason to stop verifying it.

The four measurements are recorded in centimetres:

| Column | Meaning |
|---|---|
| `sepal_length_cm` | Length of the sepal (the outer, leaf-like structure) |
| `sepal_width_cm` | Width of the sepal |
| `petal_length_cm` | Length of the petal |
| `petal_width_cm` | Width of the petal |
| `species` | Iris species label: setosa, versicolor, or virginica |

In [14]:
DATA_PATH = "iris.csv"

MEASUREMENTS = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]
COLUMN_NAMES = MEASUREMENTS + ["species"]


def load_dataset(path=DATA_PATH):
    """Read the Iris CSV into a DataFrame without modifying any values.

    Parsing is kept separate from cleaning so that the untouched input is
    available for comparison against the cleaned output.

    Parameters
    ----------
    path : str
        Path to the Iris CSV file, which must have a header row.

    Returns
    -------
    pandas.DataFrame
        The dataset exactly as stored on disk.
    """
    return pd.read_csv(path)


df_raw = load_dataset()
df_raw.head()

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [15]:
print(f"Rows:    {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")
print()
df_raw.info()

Rows:    150
Columns: 5

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sepal_length_cm  150 non-null    float64
 1   sepal_width_cm   150 non-null    float64
 2   petal_length_cm  150 non-null    float64
 3   petal_width_cm   150 non-null    float64
 4   species          150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


### 2.1 Optional: verifying the local copy against the source

The function below re-downloads the published file, checks it against a recorded
SHA-256 digest, and rewrites `iris.csv` from it. **It is deliberately not called
anywhere in this notebook**, so the analysis stays offline and deterministic;
it exists so that the committed copy can be audited on demand rather than
trusted indefinitely.

The digest is recorded rather than merely computed, which is what makes the
check meaningful. If the download does not match, the function refuses to
overwrite the local file instead of quietly replacing good data with something
unverified. Should UCI ever legitimately revise the dataset, the recorded digest
should be updated as a deliberate, reviewable edit rather than the check being
removed.

To run it, execute `refresh_from_source()` in a new cell.

In [16]:
SOURCE_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
)

# SHA-256 of the UCI file that iris.csv was built from.
SOURCE_SHA256 = (
    "6f608b71a7317216319b4d27b4d9bc84e6abd734eda7872b71a458569e2656c0"
)


def refresh_from_source(url=SOURCE_URL, expected_sha256=SOURCE_SHA256,
                        path=DATA_PATH):
    """Re-download the published dataset and rewrite the local CSV from it.

    Not called during the analysis; the notebook runs offline from the committed
    copy. Call this manually to confirm that the local file still matches what
    the UCI repository publishes, or to update it if the published data is
    revised.

    The published file has no header row, so column names are applied before the
    CSV is written. The download is rejected if its digest does not match
    ``expected_sha256``, so a verified local copy is never overwritten by
    unverified bytes.

    Parameters
    ----------
    url : str
        Address of the headerless Iris data file at the UCI repository.
    expected_sha256 : str
        Digest the download must match.
    path : str
        Local CSV to rewrite once the download has been verified.

    Returns
    -------
    pandas.DataFrame
        The freshly downloaded dataset.

    Raises
    ------
    ValueError
        If the download does not match the recorded digest. The local file is
        left untouched.
    """
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = response.read()

    digest = hashlib.sha256(payload).hexdigest()
    if digest != expected_sha256:
        raise ValueError(
            "The published file does not match the recorded checksum.\n"
            f"  expected {expected_sha256}\n"
            f"  received {digest}\n"
            f"Refusing to overwrite '{path}'."
        )

    frame = pd.read_csv(
        io.StringIO(payload.decode("utf-8")),
        header=None,
        names=COLUMN_NAMES,
    )
    frame.to_csv(path, index=False)
    print(f"Verified {len(payload)} bytes from UCI.")
    print(f"Rewrote '{path}' with {len(frame)} rows.")
    return frame

In [17]:
#refresh_from_source()

## 3. Data cleaning

Iris is often described as a dataset with "nothing to clean," and it is true that
it has no missing values. Auditing it properly turns up three real problems
anyway, and each one calls for a different response.

**Species labels carry a redundant prefix.** Every label reads `Iris-setosa`
rather than `setosa`. This is cosmetic, but it makes legends and grouped tables
noisier than they need to be.

**Two rows are known transcription errors.** The documentation shipped with the
dataset (`iris/iris.names`) records that samples 35 and 38 were transcribed
incorrectly from Fisher's original paper, and states the correct values. In the
file both rows read `4.9, 3.1, 1.5, 0.1`, which makes them look like a duplicate
pair. They are not duplicates; they are two distinct flowers, one of which was
copied wrongly. The corrected values are also distributed as
`iris/bezdekIris.data`, which lets the fix be verified rather than trusted.

**One row is a genuine duplicate.** After those corrections, sample 143 remains
an exact copy of sample 102. Both are real virginica flowers that happen to share
all four measurements to the recorded precision. Dropping it would discard a
legitimate observation and shrink a class that is supposed to contain exactly 50.

The distinction between the second and third problems drives the design below:
the audit function *reports* duplicates but never removes them, because whether a
repeated row is an error or a coincidence is a question about the world, not
about the data. Deciding that automatically is how a cleaning script quietly
introduces bias.

In [18]:
def clean_species_labels(df):
    """Normalise the species column into tidy, ordered categories.

    Strips surrounding whitespace, removes the redundant ``Iris-`` prefix,
    lowercases the result, and converts the column to an ordered categorical so
    that every grouped table and figure lists the species in the same sequence.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing a ``species`` column.

    Returns
    -------
    pandas.DataFrame
        A copy with ``species`` normalised; the input is left unchanged.

    Raises
    ------
    ValueError
        If normalisation yields a label outside the three expected species,
        which would mean the input is not the dataset this pipeline expects.
    """
    out = df.copy()
    out["species"] = (
        out["species"]
        .astype("string")
        .str.strip()
        .str.replace(r"^Iris-", "", regex=True)
        .str.lower()
    )

    unexpected = set(out["species"].dropna().unique()) - set(SPECIES_ORDER)
    if unexpected:
        raise ValueError(f"Unexpected species labels: {sorted(unexpected)}")

    out["species"] = pd.Categorical(
        out["species"], categories=SPECIES_ORDER, ordered=True
    )
    return out

In [19]:
# Corrections published in iris/iris.names, keyed by 1-based sample number.
# "corrupt" is the value as distributed; "correct" is the documented true value.
KNOWN_ERRATA = {
    35: {"corrupt": (4.9, 3.1, 1.5, 0.1), "correct": (4.9, 3.1, 1.5, 0.2)},
    38: {"corrupt": (4.9, 3.1, 1.5, 0.1), "correct": (4.9, 3.6, 1.4, 0.1)},
}


def correct_known_errata(df, errata=KNOWN_ERRATA):
    """Apply the transcription corrections documented by the dataset authors.

    Samples 35 and 38 were copied incorrectly from Fisher's original paper. The
    corrected values are published in ``iris/iris.names`` and in the alternative
    distribution ``iris/bezdekIris.data``. Each row is verified to still hold the
    known-bad values before it is overwritten, so the function fails loudly if it
    is ever pointed at data it does not actually describe, instead of silently
    corrupting good rows.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset in original file order, indexed 0..n-1.
    errata : dict
        Maps a 1-based sample number to its ``corrupt`` and ``correct``
        four-measurement tuples.

    Returns
    -------
    pandas.DataFrame
        A copy with the documented corrections applied.

    Raises
    ------
    ValueError
        If a targeted row does not contain the expected incorrect values.
    """
    out = df.copy()

    for sample_no, fix in errata.items():
        idx = sample_no - 1
        found = tuple(out.loc[idx, MEASUREMENTS].astype(float))

        if found != fix["corrupt"]:
            raise ValueError(
                f"Sample {sample_no} holds {found}, expected the documented "
                f"incorrect values {fix['corrupt']}. Refusing to overwrite."
            )

        out.loc[idx, MEASUREMENTS] = fix["correct"]
        print(f"Sample {sample_no}: {found} -> {fix['correct']}")

    return out

In [20]:
def audit_measurements(df, columns=MEASUREMENTS):
    """Coerce measurements to numeric and report data-quality problems.

    Casts the measurement columns to float, then checks three things that would
    invalidate the analysis: missing values, physically impossible measurements
    (a length must be greater than zero), and exact duplicate rows.

    Duplicates are reported but deliberately **not** removed. Whether a repeated
    row is a recording error or two genuinely identical specimens cannot be
    decided from the table alone, so the judgement is left to the analyst and
    recorded in prose.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset with the measurement columns present.
    columns : list of str
        Measurement columns to validate.

    Returns
    -------
    (pandas.DataFrame, dict)
        The dataset with numeric measurement columns, and a report with keys
        ``missing``, ``non_positive`` and ``duplicate_rows``.

    Raises
    ------
    ValueError
        If any measurement is missing or non-positive, which would mean the file
        needs repair before analysis can proceed.
    """
    out = df.copy()
    out[columns] = out[columns].apply(pd.to_numeric, errors="coerce")

    missing = out[columns].isna().sum()
    non_positive = int((out[columns] <= 0).to_numpy().sum())
    duplicate_mask = out.duplicated(keep=False)

    report = {
        "missing": missing,
        "non_positive": non_positive,
        "duplicate_rows": out.index[duplicate_mask].tolist(),
    }

    print("Missing values per column:")
    print(missing.to_string())
    print(f"\nNon-positive measurements: {non_positive}")
    print(f"Exact duplicate rows (0-based index): {report['duplicate_rows']}")

    if missing.sum() or non_positive:
        raise ValueError("Dataset contains missing or non-positive measurements.")

    return out, report

### Running the cleaning pipeline

The three functions are composed in order: fix the labels, repair the documented
errata, then audit what remains. Each returns a new frame rather than mutating
its input, so `df_raw` stays available for the before-and-after comparison below.

In [21]:
df_labelled = clean_species_labels(df_raw)
df_corrected = correct_known_errata(df_labelled)
df, quality_report = audit_measurements(df_corrected)

print(f"\nCleaned shape: {df.shape}")
df.head()

Sample 35: (4.9, 3.1, 1.5, 0.1) -> (4.9, 3.1, 1.5, 0.2)
Sample 38: (4.9, 3.1, 1.5, 0.1) -> (4.9, 3.6, 1.4, 0.1)
Missing values per column:
sepal_length_cm    0
sepal_width_cm     0
petal_length_cm    0
petal_width_cm     0

Non-positive measurements: 0
Exact duplicate rows (0-based index): [101, 142]

Cleaned shape: (150, 5)


,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [22]:
print("Species labels before cleaning:")
print(df_raw["species"].value_counts().to_string())

print("\nSpecies labels after cleaning:")
print(df["species"].value_counts().sort_index().to_string())

print("\nRows 35 and 38, before and after the errata correction:")
comparison = pd.concat(
    [
        df_raw.loc[[34, 37], MEASUREMENTS].add_suffix(" (raw)"),
        df.loc[[34, 37], MEASUREMENTS].add_suffix(" (fixed)"),
    ],
    axis=1,
)
comparison.index = ["sample 35", "sample 38"]
comparison.T

Species labels before cleaning:
species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50

Species labels after cleaning:
species
setosa        50
versicolor    50
virginica     50

Rows 35 and 38, before and after the errata correction:


,sample 35,sample 38
sepal_length_cm (raw),4.9,4.9
sepal_width_cm (raw),3.1,3.1
petal_length_cm (raw),1.5,1.5
petal_width_cm (raw),0.1,0.1
sepal_length_cm (fixed),4.9,4.9
sepal_width_cm (fixed),3.1,3.6
petal_length_cm (fixed),1.5,1.4
petal_width_cm (fixed),0.2,0.1


### What the audit found

The audit confirms no missing values and no impossible measurements, so no rows
were imputed or discarded. The only remaining duplicate is sample 143, an exact
copy of sample 102. It is kept, for the reason given above: both are real
virginica specimens, and removing one would leave that class with 49 flowers
while the other two keep 50, introducing an imbalance that the original data does
not have. The analysis therefore runs on all 150 rows.

In [23]:
duplicate_idx = quality_report["duplicate_rows"]
print(f"Retained duplicate pair (0-based index {duplicate_idx}):")
df.loc[duplicate_idx]

Retained duplicate pair (0-based index [101, 142]):


,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
101,5.8,2.7,5.1,1.9,virginica
142,5.8,2.7,5.1,1.9,virginica


## 4. Exploratory data analysis

## 5. Visualizations

## 6. Summary and interpretation